# nanoGPT Training (Andrej Model + DIY Data)

This notebook uses Andrej Karpathy's **nanoGPT** model, optimizer, loss function, and default
training parameter settings from `nanoGPT/train.py`, but points the data loader at
`DIY_nanoGPT/data/tokenized/concatenated` (or `tokenized_concatenated` if present).


In [19]:
from google.colab import drive
drive.mount('COLAB_DRIVE')

Drive already mounted at COLAB_DRIVE; to attempt to forcibly remount, call drive.mount("COLAB_DRIVE", force_remount=True).


In [20]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import tiktoken

import numpy as np

from beartype import beartype
import json
import glob

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

project_name = "nanoGPT" # variable depends on project

if is_colab():
    matches = glob.glob(f"COLAB_DRIVE/MyDrive/**/{project_name}/", recursive=True)
    print(matches)
    if len(matches) == 1:
        proj_dir = matches[0]
    with open(os.path.join(proj_dir, "config.json"), "r") as f:
        config = json.load(f)
    config['seed_settings']['use_seed'] = False

else:
    from root_path import get_root_path
    proj_dir = get_root_path()
    print("Is local")
    with open(os.path.join(proj_dir, "config.json"), "r") as f:
        config = json.load(f)

import sys
sys.path.append(proj_dir)

from utils import torch_ckpt
from model import model

['COLAB_DRIVE/MyDrive/Project/nanoGPT/']


In [21]:
# Environment + path setup
import os
import sys
import time
import math
import pickle
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import torch

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'DIY_nanoGPT').exists() and (p / 'nanoGPT').exists():
            return p
    raise FileNotFoundError('Could not locate repo root containing DIY_nanoGPT/ and nanoGPT/')

diy_dir = Path(proj_dir)
andrej_dir = Path(proj_dir) / 'Trainer' / 'Andrej_file'

sys.path.insert(0, str(andrej_dir))
from Andrej_model import GPTConfig, GPT

print('diy_dir:', diy_dir)
print('andrej_dir:', andrej_dir)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())


diy_dir: COLAB_DRIVE/MyDrive/Project/nanoGPT
andrej_dir: COLAB_DRIVE/MyDrive/Project/nanoGPT/Trainer/Andrej_file
torch: 2.9.0+cu126
cuda available: True


In [22]:
# Andrej Karpathy default settings (from nanoGPT/train.py)
# --- Modified to match DIY_nanoGPT Trainer.ipynb memory footprint ---
# I/O
out_dir = str(diy_dir / 'ckpt_andrej_nanoGPT')
eval_interval = 2000
log_interval = 1
eval_iters = 200
eval_only = False
always_save_checkpoint = True
init_from = 'scratch'  # 'scratch' | 'resume' | 'gpt2*'

# Data
gradient_accumulation_steps = 1
batch_size = 4
block_size = 1024

# Model
n_layer = 12
n_head = 12
n_embd = 768
dropout = 0.0
bias = False

# AdamW optimizer
learning_rate = 6e-4
max_iters = 600000
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0

# Learning rate decay
decay_lr = True
warmup_iters = 2000
lr_decay_iters = 600000
min_lr = 6e-5

# System
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
compile = True  # PyTorch 2.0+

tokens_per_iter = gradient_accumulation_steps * batch_size * block_size
print(f'tokens per iteration: {tokens_per_iter:,}')
print(f'device={device}, dtype={dtype}, compile={compile}')
os.makedirs(out_dir, exist_ok=True)
torch.manual_seed(1337)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

tokens per iteration: 4,096
device=cuda, dtype=bfloat16, compile=True


In [23]:
# Model config toggle
# True: use a custom nanoGPT config that matches DIY_nanoGPT model settings
# False: use the original nanoGPT 124M config (GPT-2 small)
use_custom_diy_config = True

custom_diy_config = dict(
    n_layer=4,
    n_head=6,
    n_embd=768,
    block_size=1024,
    dropout=0.1,
    bias=False,
    vocab_size=57628,
)


In [24]:
# Data loader (memmap)
data_dir_candidates = [
    diy_dir / 'data' / 'tokenized_concatenated',
    diy_dir / 'data' / 'tokenized' / 'concatenated',
]
data_dir = None
for cand in data_dir_candidates:
    if cand.exists():
        data_dir = cand
        break
if data_dir is None:
    raise FileNotFoundError(f'No data directory found. Checked: {data_dir_candidates}')

train_bin = data_dir / 'train_all.bin'
val_bin = data_dir / 'val_all_10_to_11.bin'
if not train_bin.exists():
    train_bin = data_dir / 'train.bin'
if not val_bin.exists():
    val_bin = data_dir / 'val.bin'

print('data_dir:', data_dir)
print('train_bin:', train_bin)
print('val_bin:', val_bin)

def get_batch(split):
    if split == 'train':
        data = np.memmap(train_bin, dtype=np.uint16, mode='r')
    else:
        data = np.memmap(val_bin, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

# Optional vocab size discovery
meta_path = data_dir / 'meta.pkl'
meta_vocab_size = None
if meta_path.exists():
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    meta_vocab_size = meta.get('vocab_size')
    print(f'found vocab_size={meta_vocab_size} in {meta_path}')


data_dir: COLAB_DRIVE/MyDrive/Project/nanoGPT/data/tokenized/concatenated
train_bin: COLAB_DRIVE/MyDrive/Project/nanoGPT/data/tokenized/concatenated/train_all.bin
val_bin: COLAB_DRIVE/MyDrive/Project/nanoGPT/data/tokenized/concatenated/val_all_10_to_11.bin


In [25]:
# Model init (nanoGPT GPT)
# Switch between DIY-like custom config and original nanoGPT 124M config
if use_custom_diy_config:
    model_args = dict(
        n_layer=custom_diy_config['n_layer'],
        n_head=custom_diy_config['n_head'],
        n_embd=custom_diy_config['n_embd'],
        block_size=custom_diy_config['block_size'],
        bias=custom_diy_config['bias'],
        vocab_size=custom_diy_config['vocab_size'],
        dropout=custom_diy_config['dropout'],
    )
    init_from = 'scratch'
    gptconf = GPTConfig(**model_args)
    model = GPT(gptconf)
else:
    # Original nanoGPT 124M defaults
    model_args = dict(
        n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,
        bias=bias, vocab_size=None, dropout=dropout
    )

    if init_from == 'scratch':
        if meta_vocab_size is None:
            print('defaulting vocab_size to 50304 (GPT-2 padded)')
        model_args['vocab_size'] = meta_vocab_size if meta_vocab_size is not None else 50304
        gptconf = GPTConfig(**model_args)
        model = GPT(gptconf)
    elif init_from.startswith('gpt2'):
        model = GPT.from_pretrained(init_from, dict(dropout=dropout))
        for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
            model_args[k] = getattr(model.config, k)
    else:
        raise ValueError(f'Unsupported init_from: {init_from}')

model.to(device)

scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)

if compile:
    print('compiling model (PyTorch 2.0+)')
    model = torch.compile(model)


number of parameters: 72.58M
num decayed parameter tensors: 18, with 73,356,288 parameters
num non-decayed parameter tensors: 9, with 6,912 parameters
using fused AdamW: True
compiling model (PyTorch 2.0+)


/tmp/ipython-input-191823835.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


In [26]:
# Loss estimation
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Learning rate schedule (cosine with warmup)
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


In [27]:
# Training loop (single process, Andrej-style)
iter_num = 0
best_val_loss = 1e9

train_loss_list = {}
val_loss_list = {}

X, Y = get_batch('train')
start_time = time.time()

def format_elapsed(seconds):
    minutes = int(seconds // 60)
    secs = seconds - minutes * 60
    return f"{minutes}m {secs:05.2f}s"

while True:
    step_start = time.time()
    lr = get_lr(iter_num) if decay_lr else learning_rate
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    log_lines = []

    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        log_lines.append(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        if losses['val'] < best_val_loss or always_save_checkpoint:
            best_val_loss = losses['val']
            if iter_num > 0:
                ckpt = {
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'model_args': model_args,
                    'iter_num': iter_num,
                    'best_val_loss': best_val_loss,
                }
                torch.save(ckpt, os.path.join(out_dir, 'ckpt.pt'))
                log_lines.append(f'saving checkpoint to {out_dir}')
    if iter_num == 0 and eval_only:
        break

    for micro_step in range(gradient_accumulation_steps):
        with ctx:
            _, loss = model(X, Y)
            loss = loss / gradient_accumulation_steps
        X, Y = get_batch('train')
        scaler.scale(loss).backward()

    if grad_clip != 0.0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    lossf = loss.item() * gradient_accumulation_steps
    train_loss_list[iter_num] = {"loss": lossf, "batch_size": batch_size}

    with torch.no_grad():
        model.eval()
        Xv, Yv = get_batch('val')
        with ctx:
            _, val_loss = model(Xv, Yv)
        model.train()
    val_loss_list[iter_num] = {"loss": val_loss.item(), "batch_size": batch_size}

    now = time.time()
    step_time = now - step_start
    elapsed_time = now - start_time
    time_suffix = f" | step {format_elapsed(step_time)} | elapsed {format_elapsed(elapsed_time)}"

    for line in log_lines:
        print(line + time_suffix)

    print(f'iter {iter_num}: train loss {lossf:.4f}, val loss {val_loss.item():.4f}' + time_suffix)

    iter_num += 1
    if iter_num > max_iters:
        break


step 0: train loss 11.1146, val loss 11.1160


/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2772: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


iter 0: loss 11.1297, time 207120.22ms
iter 1: loss 11.1115, time 37.23ms
iter 2: loss 11.0904, time 1409.09ms
iter 3: loss 11.0906, time 1836.61ms
iter 4: loss 11.0816, time 1167.32ms
iter 5: loss 11.0533, time 528.17ms
iter 6: loss 11.0193, time 826.95ms
iter 7: loss 11.0098, time 1487.02ms
iter 8: loss 10.9855, time 1184.74ms
iter 9: loss 10.9474, time 1173.06ms
iter 10: loss 10.8777, time 1242.64ms
iter 11: loss 10.8723, time 548.09ms
iter 12: loss 10.8405, time 1472.94ms
iter 13: loss 10.7948, time 1212.78ms
iter 14: loss 10.7723, time 559.16ms
iter 15: loss 10.7011, time 830.61ms
iter 16: loss 10.6503, time 1501.35ms
iter 17: loss 10.6617, time 544.85ms
iter 18: loss 10.5759, time 846.63ms
iter 19: loss 10.5517, time 1437.61ms
iter 20: loss 10.4547, time 554.68ms
iter 21: loss 10.4777, time 838.60ms
iter 22: loss 10.3325, time 1487.27ms
iter 23: loss 10.4496, time 1265.79ms
iter 24: loss 10.3234, time 555.67ms
iter 25: loss 10.2106, time 832.98ms
iter 26: loss 10.3057, time 1567.

KeyboardInterrupt: 

Why does logging the loss value slower than my code?<br>
What's the settings that Andrej did?